In [ ]:
class WithDictionary:
    def __init__(self):
        self.volumes: dict[int, list[int]] = {-1:[], 1:[]}
            
class Elementwise:
    def __init__(self):
        self.ask_volumes: list[int] = []
        self.bid_volumes: list[int] = []
            
    def get_volumes(self, direction:int) -> list[int]:
        if direction == -1:
            return self.ask_volumes
        else:
            return self.bid_volumes
        
class WithTuple:
    def __init__(self):
        self.volumes: tuple[list[int], list[int]] = ([], [])
        # self.volumes[0] is the ask side
        # self.volumes[1] is the bid sie
    def get_volumes(self, direction:int) -> list[int]:
        return self.volumes[(1 + direction) // 2]

In [ ]:
with_dict = WithDictionary()
elementwise = Elementwise()
withtuple = WithTuple()

In [ ]:
%%timeit -n 1000 -r 1000
with_dict.volumes[1]

In [ ]:
%%timeit -n 1000 -r 1000
elementwise.get_volumes(1)

In [ ]:
%%timeit -n 1000 -r 1000
elementwise.bid_volumes

In [ ]:
%%timeit -n 1000 -r 1000
withtuple.get_volumes(1)

---

## The comparison above is not like for like

`with_dict.volumes[1]` is a subscript; `elementwise.get_volumes(1)` is a **method call**.
Most of the 10 ns between them is the call, not the branch — so the measurement does not
say that a direction-keyed dict beats branching on direction. It says that calling a
method costs something, which we already knew.

To ask the real question, put both sides inside a method, since that is where the code
that matters lives: `AggregateBook` and its variants are generic in `d` and reach for the
side from inside `best_price` and `set_volume`.

In [ ]:
class WithDict:
    def __init__(self):
        self.by_direction = {1: 'bid', -1: 'ask'}
    def get(self, direction):
        return self.by_direction[direction]

class WithBranch:
    def __init__(self):
        self.bid, self.ask = 'bid', 'ask'
    def get(self, direction):
        return self.bid if direction == 1 else self.ask

class WithList:
    def __init__(self):
        # Indexed by direction directly: [1] is the bid, [-1] the ask.  Three slots,
        # not two -- in a two-element list [1] and [-1] are the same slot.
        self.by_direction = [None, 'bid', 'ask']
    def get(self, direction):
        return self.by_direction[direction]

with_dict, with_branch, with_list = WithDict(), WithBranch(), WithList()
assert (with_dict.get(-1), with_branch.get(-1), with_list.get(-1)) == ('ask',) * 3
assert (with_dict.get(1), with_branch.get(1), with_list.get(1)) == ('bid',) * 3

In [ ]:
%%timeit -n 100000 -r 20
with_dict.get(-1)

In [ ]:
%%timeit -n 100000 -r 20
with_branch.get(-1)

In [ ]:
%%timeit -n 100000 -r 20
with_list.get(-1)

The three land within a few nanoseconds of each other, and the ordering is not stable
across runs. So this is not a performance decision at all.

`unito26.lob.orderbook` keeps the dict, for a reason that has nothing to do with speed:
the code around it is written generically in `d`, and a dict keyed by `d` says that
plainly where a branch restates the two cases every time it is read. The list is the
fastest of the three and the worst to read — `[None, 'bid', 'ask']` needs a comment to
explain a hole in position zero.

Worth keeping as an exam snippet: *given these two timings, what may you conclude?*